In [ ]:
from typing import Literal,TypedDict
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from pydantic import BaseModel
from langgraph.graph import state, START,END
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
import db
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request
import os


## Loading API Key

In [2]:
load_dotenv()

True

## Loading Model

In [3]:
model = ChatGroq(model = 'llama-3.3-70b-versatile')

## LogIN Manager 

In [ ]:
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly', 'https://www.googleapis.com/auth/gmail.send'] # gmail ko read krne k lye aur send krne k lye api ko use krna hai 
def get_gmail_creds():
    creds = None
    if not creds or not creds.valid: # agr credentials valid nahi hai to phir check kro k wo expired hai ya nahi aur refresh token available hai ya nahi
        if creds and creds.expired and creds.refresh_token: # agr credentials expired hain to phir refresh kro
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("credentials.json", SCOPES) # agr credential create hi nhi hue 
            creds = flow.run_local_server(port=0)
        with open("token.json", "w") as f:
            f.write(creds.to_json()) # credentials ko save kro

    return creds

IndentationError: expected an indented block after 'else' statement on line 7 (1936513629.py, line 7)

## State of Graph

In [4]:
class State(TypedDict):
    client_id : int
    name : str
    follow_up_count: int
    email_subject: str
    message_draft : str
    client_history : dict
   
    
    

## Connecting to database

In [5]:
conn = sqlite3.connect('followup_agent.db',check_same_thread=False)
checkpointer = SqliteSaver(conn)

## Node1 - Cotext Loader

In [8]:
def context_loader(State)->dict:
    client = db.get_client(state['client_id'])
    history = db.get_client_history(state['client_id'],limit=10)
    return {
        "client_name": client['name'],
        "client_history": history}
    

c:\Users\Amna Ali\AppData\Local\Programs\Python\Python310\python.exe
